# Problem Book P1, Chapter 10: Fourier at the Crossroads (Python companion)

Part A checks the numbers of the chapter, problem by problem. Part B is a laboratory: four experiments that push the
problems further, with exercises first and solutions at the end. Only `numpy`, `mpmath` and `matplotlib` are needed.

*Before running a cell, predict its output.*

In [ ]:
import math, random, itertools, cmath
from math import comb, factorial, log, sqrt, pi, asin, cos, sin
from fractions import Fraction as F
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt
mp.mp.dps = 30
random.seed(1); rng = np.random.default_rng(1)
def e(x): return np.exp(2j * np.pi * x)
def legendre_symbol(a, p): a %= p; return 0 if a == 0 else (1 if pow(a, (p - 1) // 2, p) == 1 else -1)

## Part A: checks

**Problem 10.1 (sums over boxes).** The bilinear bound for random sets, the incidence bound (b) for random $U,V\subseteq\mathbb F_p^2$, and
$ab+cd$ covering $\mathbb F_p^\times$ for a set of size just above $p^{3/4}$.

In [ ]:
p = 211
A = rng.choice(p, 40, replace=False); B = rng.choice(p, 60, replace=False)
print(max(abs(e((r * np.outer(A, B) % p) / p).sum()) for r in range(1, p)), sqrt(p * 40 * 60))
U = rng.integers(0, p, (300, 2)); V = rng.integers(0, p, (300, 2))
dots = (U @ V.T) % p
dev = max(abs(np.sum(dots == x) - 300 * 300 / p) for x in range(1, p))
print(round(dev, 1), round(sqrt(p * 300 * 300), 1))
m = int(p**0.75) + 1; A = rng.choice(np.arange(1, p), m, replace=False)
prods = np.unique(np.outer(A, A) % p); sums = np.unique((prods[:, None] + prods[None, :]) % p)
print(m, p**0.75, set(range(1, p)) <= set(sums.tolist()))

**Problem 10.2 (Kloosterman).** Second and fourth moments, and the bounds $1.19p^{3/4}$ (Kloosterman) and $2\sqrt p$ (Weil).

In [ ]:
def kloosterman(p):
    inv = np.array([0] + [pow(x, p - 2, p) for x in range(1, p)]); xs = np.arange(1, p)
    return np.array([np.cos(2 * np.pi * ((a * xs + inv[xs]) % p) / p).sum() for a in range(p)])
for p in (7, 13, 101, 401):
    K = kloosterman(p)
    print(p, round(K[0], 6), round((K**2).sum()), p * p - p, round((K**4).sum()), 2 * p**3 - 3 * p * p - 3 * p,
          round(abs(K[1:]).max() / sqrt(p), 3), round(1.19 * p**0.75 / sqrt(p), 2))

**Problem 10.3 (progressions).** $T(Q)$ for the quadratic residues against $\frac{(p-1)^3}{8p}$ and the error bound.

In [ ]:
for p in (101, 409, 1009):
    Q = np.zeros(p, bool); Q[(np.arange(1, p)**2) % p] = True
    x = np.arange(p); T = sum(int(np.sum(Q & Q[(x + d) % p] & Q[(x + 2 * d) % p])) for d in range(p))
    print(p, T, round((p - 1)**3 / (8 * p), 1), round(abs(T - (p - 1)**3 / (8 * p)), 1), round((sqrt(p) + 1) * (p - 1) / 4, 1))

**Problem 10.4 (DFT eigenvalues).** Multiplicities of $1,-1,-i,i$ against the formula.

In [ ]:
def predicted(n):
    m, s = divmod(n, 4)
    return [(m + 1, m, m, m - 1), (m + 1, m, m, m), (m + 1, m + 1, m, m), (m + 1, m + 1, m + 1, m)][s]
for n in range(2, 17):
    Fm = np.array([[cmath.exp(-2j * pi * j * k / n) for k in range(n)] for j in range(n)]) / sqrt(n)
    ev = np.linalg.eigvals(Fm)
    got = tuple(int(np.sum(abs(ev - v) < 1e-6)) for v in (1, -1, -1j, 1j))
    assert got == predicted(n), n
print("multiplicities agree for n = 2..16; trace of F for n = 4..7:",
      [complex(np.round(np.trace(np.array([[cmath.exp(-2j * pi * j * k / n) for k in range(n)] for j in range(n)]) / sqrt(n)), 6)) for n in (4, 5, 6, 7)])

**Problem 10.5 (Newman).** The counts of digit sums modulo $3$, and the two exact biases.

In [ ]:
s = lambda n: bin(n).count("1")
for k in (5, 8, 11):
    cnt = [sum(1 for n in range(2**k) if s(n) % 3 == j) for j in range(3)]
    print(k, cnt, [round((2**k + 2 * cos(pi * (k - 2 * j) / 3)) / 3) for j in range(3)])
print([(k, sum((-1)**s(n) for n in range(0, 4**k, 3)), 2 * 3**(k - 1), sum((-1)**s(n) for n in range(0, 2 * 4**k, 3)), 3**k) for k in range(1, 8)])
S = np.cumsum([(-1)**s(n) if n % 3 == 0 else 0 for n in range(1, 2**16)])
print("positive for every N up to 2^16:", (S + 1 > 0).all())

**Problem 10.6 (discrepancy of $k^2/p$).** The star discrepancy against $\frac{\log p+10}{\pi\sqrt p}$.

In [ ]:
for p in (101, 1009, 10007, 100003):
    pts = np.sort((np.arange(p, dtype=np.int64)**2 % p) / p); i = np.arange(p)
    D = max(np.max((i + 1) / p - pts), np.max(pts - i / p))
    print(p, round(D, 5), round((log(p) + 10) / (pi * sqrt(p)), 5), round(D * sqrt(p) / log(p), 3))

**Problem 10.7 (walk on the circle).** $|\mathbb E\,e(hX_n)|=|\cos(2\pi h\alpha)|^n$ for $\alpha=\sqrt2$, and the two alternating limits for $\alpha=\frac14$.

In [ ]:
print([round(abs(cos(2 * pi * h * sqrt(2)))**200, 4) for h in range(1, 6)])
def law(n, q, a=1):                     # exact law of X_n on (1/q)Z/Z for alpha = a/q
    pr = np.zeros(q); pr[0] = 1
    for _ in range(n): pr = 0.5 * (np.roll(pr, a) + np.roll(pr, -a))
    return pr
print(np.round(law(200, 4), 4), np.round(law(201, 4), 4), np.round(law(200, 5), 4))

**Problem 10.8 (lacunary sums).** Exact fourth moments by an FFT-exact average over $2^{N+3}$ points, against $\frac34N^2+\frac98N-3$, and $\mathbb E[T_N^4]$
against its exact count.

In [ ]:
for N in (2, 4, 8, 12):
    M = 2**(N + 3); t = 2 * np.pi * np.arange(M) / M
    S = sum(np.cos(2.0**k * t) for k in range(1, N + 1)); T = sum(np.cos(k * t) for k in range(1, N + 1))
    exactT = (2 * (2 * N**3 + N) + 8 * comb(N, 3)) / 16
    print(N, round(np.mean(S**4), 6), 0.75 * N * N + 9 * N / 8 - 3, round(np.mean(T**4), 6), exactT, round(N**3 / 3, 1))

**Problem 10.9 (the Cantor law).** $|\varphi(2\pi3^m)|$ is constant, while Viète's product gives $\frac{\sin t}t$.

In [ ]:
def phi_cantor(t, K=60): return abs(mp.fprod(mp.cos(t / mp.mpf(3)**k) for k in range(1, K)))
print([mp.nstr(phi_cantor(2 * mp.pi * 3**m), 10) for m in range(6)], mp.nstr(mp.exp(-mp.pi**2 / 18) / 2, 6))
t = mp.mpf(5); print(mp.fprod(mp.cos(t / 2**k) for k in range(1, 80)), mp.sin(t) / t)

**Problem 10.10 (sinc sums).** The sum and the integral of $\operatorname{sinc}^k$ agree for $k\le6$ and part at $k=7$.

In [ ]:
def g(k, s):                            # density of U_1+...+U_k, U_i uniform on [-1,1]
    return sum((-1)**j * comb(k, j) * max(s + k - 2 * j, 0)**(k - 1) for j in range(k + 1)) / (2**k * factorial(k - 1))
for k in range(2, 9):
    integral = 2 * mp.pi * g(k, 0)
    direct = 1 + 2 * mp.fsum(mp.sinc(n)**k for n in range(1, 100000)) if k >= 3 else None
    poisson = integral + 2 * mp.pi * sum(2 * g(k, 2 * mp.pi * m) for m in range(1, 3))
    print(k, mp.nstr(integral, 14), mp.nstr(poisson, 14), direct and mp.nstr(direct, 14))
print(mp.nstr(4 * mp.pi * (7 - 2 * mp.pi)**6 / (2**7 * factorial(6)), 6))
print("k = 1:", mp.nstr(mp.nsum(lambda n: mp.sin(n) / n, [1, mp.inf]), 12), mp.nstr((mp.pi - 1) / 2, 12))

For the cube problems we use the Walsh--Hadamard transform: `fourier(f)` returns all $2^n$ coefficients $\hat f(S)$, indexed by bit masks.

In [ ]:
def fourier(vals):                      # vals[x] for x in 0..2^n-1, bit i = 1 meaning x_i = -1
    a = np.array(vals, dtype=float).copy(); h = 1
    while h < len(a):
        for i in range(0, len(a), 2 * h):
            u, v = a[i:i + h].copy(), a[i + h:i + 2 * h].copy(); a[i:i + h], a[i + h:i + 2 * h] = u + v, u - v
        h *= 2
    return a / len(a)
def boolfun(n, rule): return [rule(tuple(1 - 2 * ((x >> i) & 1) for i in range(n))) for x in range(2**n)]
maj = lambda x: 1 if sum(x) > 0 else -1
popcount = np.vectorize(lambda m: bin(m).count("1"))

**Problem 10.11 (influences).** Influences of majority from the spectrum, against $\binom{n-1}{(n-1)/2}2^{1-n}$, and $W^1\to\frac2\pi$.

In [ ]:
for n in (3, 5, 9, 13):
    c = fourier(boolfun(n, maj)); deg = popcount(np.arange(2**n))
    inf1 = sum(c[m]**2 for m in range(2**n) if m & 1)
    print(n, round(inf1, 6), comb(n - 1, (n - 1) // 2) / 2**(n - 1), round((deg * c**2).sum(), 4), round(sqrt(2 * n / pi), 4), round(c[deg == 1].__pow__(2).sum(), 4))
print(2 / pi)

**Problem 10.12 (granularity).** For random Boolean functions of a few variables, the coefficients are multiples of $2^{1-d}$; the address function has degree
$d$ and $d-1+2^{d-1}$ relevant variables.

In [ ]:
for trial in range(4):
    n = 4; vals = rng.choice([-1, 1], 2**n); c = fourier(vals); deg = popcount(np.arange(2**n))
    d = max(deg[np.abs(c) > 1e-12]); q = c * 2**(d - 1)
    print(d, np.allclose(q, np.round(q)), int(np.sum(np.abs(c) > 1e-12)), 4**(d - 1))
d = 3; n = d - 1 + 2**(d - 1)
addr = lambda x: x[d - 1 + ((1 - x[0]) // 2) + 2 * ((1 - x[1]) // 2)]
c = fourier(boolfun(n, addr)); deg = popcount(np.arange(2**n))
print("address:", n, "variables, degree", max(deg[np.abs(c) > 1e-12]))

**Problem 10.13 (bent functions).** $(-1)^{x\cdot y}$ on $\mathbb F_2^6$: all $|\hat f|=2^{-3}$, nonlinearity $2^5-2^2=28$, and $36$ or $28$ values $-1$.

In [ ]:
m = 3; n = 2 * m
f = [(-1)**bin((x & (2**m - 1)) & (x >> m)).count("1") for x in range(2**n)]
c = fourier(f)
print(set(np.round(np.abs(c), 6)), 2**(n - 1) * (1 - np.abs(c).max()), f.count(-1), 2**(n - 1) - 2**(m - 1))

**Problem 10.14 (noise stability).** $\sum_S\rho^{|S|}\hat f(S)^2$ for majority, against $\frac2\pi\arcsin\rho$, and the limits of $W^1$, $W^3$.

In [ ]:
for n in (5, 11, 15):
    c = fourier(boolfun(n, maj)); deg = popcount(np.arange(2**n))
    print(n, round(float((0.5**deg * c**2).sum()), 4), round(2 / pi * asin(0.5), 4),
          round(c[deg == 1].__pow__(2).sum(), 4), round(c[deg == 3].__pow__(2).sum(), 4))
print(2 / pi, 1 / (3 * pi))

**Problem 10.15 (Condorcet and Kalai).** Brute force over all profiles against Kalai's formula, for majority and for random rules.

In [ ]:
perms = list(itertools.permutations("abc"))
def prefer(r, u, v): return 1 if r.index(u) < r.index(v) else -1
def rational_prob(n, f):
    good = 0
    for prof in itertools.product(perms, repeat=n):
        x = tuple(prefer(r, "a", "b") for r in prof); y = tuple(prefer(r, "b", "c") for r in prof); z = tuple(prefer(r, "c", "a") for r in prof)
        good += not (f(x) == f(y) == f(z))
    return F(good, 6**n)
def kalai(n, f):
    c = fourier(boolfun(n, f)); deg = popcount(np.arange(2**n))
    return 0.75 - 0.75 * float(((-1 / 3)**deg * c**2).sum())
print(rational_prob(3, maj), F(17, 18), kalai(3, maj))
table = {x: random.choice([-1, 1]) for x in itertools.product([-1, 1], repeat=4)}
print(float(rational_prob(4, lambda x: table[x])), kalai(4, lambda x: table[x]))
print("Guilbaud:", 0.75 + 3 / (2 * pi) * asin(1 / 3))

**Problem 10.16 (Bonami).** The worst ratio $\mathbb E f^4/\mathbb E[f^2]^2$ over random degree-2 polynomials stays far below $81$; products of block sums
approach $3^d$.

In [ ]:
n = 10; X = np.array(list(itertools.product([-1, 1], repeat=n)))
worst = 0
for _ in range(300):
    pairs = list(itertools.combinations(range(n), 2)); c = rng.normal(size=len(pairs)) * (rng.random(len(pairs)) < 0.2)
    f = sum(ci * X[:, a] * X[:, b] for ci, (a, b) in zip(c, pairs))
    if np.mean(f * f) > 0: worst = max(worst, np.mean(f**4) / np.mean(f * f)**2)
blocks = (X[:, :5].sum(1) / sqrt(5)) * (X[:, 5:].sum(1) / sqrt(5))
print(round(worst, 3), round(np.mean(blocks**4) / np.mean(blocks**2)**2, 4), (3 - 2 / 5)**2)

**Problem 10.17 (cutoff).** Exact distance through the weight chain, against the bounds of (d).

In [ ]:
def tv_cube(n, t):
    p = np.zeros(n + 1); p[0] = 1; w = np.arange(n + 1)
    b = np.array([comb(n, k) for k in range(n + 1)], float) / 2**n
    for _ in range(t):
        q = 0.5 * p; q[1:] += p[:-1] * (n - w[:-1]) / (2 * n); q[:-1] += p[1:] * w[1:] / (2 * n); p = q
    return 0.5 * np.abs(p - b).sum()
n = 400
for c in (-2.5, -1.5, 0.5, 1):
    t = math.ceil(0.5 * n * log(n) + c * n) if c > 0 else math.floor(0.5 * n * log(n) + c * n)
    bound = 0.5 * sqrt(math.expm1(math.exp(-2 * c))) if c > 0 else 1 - 8 * math.exp(2 * c)   # c < 0 here means t = n log n / 2 - |c| n
    print(c, t, round(tv_cube(n, t), 4), "upper" if c > 0 else "lower", round(bound, 4))

**Problem 10.18 (non-negative trigonometric polynomials).** A Fejér--Riesz factorisation computed from the roots, Fejér's bound, and the
Egerváry--Szász extremal polynomial.

In [ ]:
c = {0: 1.0, 1: 0.6, -1: 0.6, 2: 0.25, -2: 0.25}            # T = 1 + 1.2 cos t + 0.5 cos 2t >= 0
ts = np.linspace(0, 2 * pi, 1001); T = sum(v * np.exp(1j * k * ts) for k, v in c.items()).real
roots = np.roots([c[k] for k in (2, 1, 0, -1, -2)]); inside = roots[np.abs(roots) <= 1]
P = np.poly(inside) * sqrt(abs(c[2]) / np.prod(np.abs(inside)))
print(T.min() > 0, np.abs(np.abs(np.polyval(P, np.exp(1j * ts)))**2 - T).max())
for n in (2, 5, 10):
    v = np.sin(np.arange(1, n + 2) * pi / (n + 2)); v /= np.linalg.norm(v)
    a1 = 2 * sum(v[k] * v[k + 1] for k in range(n)); Tt = np.abs(np.polyval(v[::-1], np.exp(1j * ts)))**2
    print(n, round(a1, 6), round(2 * cos(pi / (n + 2)), 6), Tt.min() >= -1e-12, round(np.abs(np.polyval(np.ones(n + 1), 1))**2 / (n + 1), 3))

**Problem 10.19 (Bernstein).** $\max|T'|/\max|T|$ for random trigonometric polynomials of degree $8$, and the Bernstein--Szegő quantity.

In [ ]:
n = 8; ts = np.linspace(0, 2 * pi, 20001); worst = 0; worst2 = 0
for _ in range(500):
    a, b = rng.normal(size=n + 1), rng.normal(size=n + 1)
    T = sum(a[k] * np.cos(k * ts) + b[k] * np.sin(k * ts) for k in range(n + 1))
    dT = sum(-k * a[k] * np.sin(k * ts) + k * b[k] * np.cos(k * ts) for k in range(n + 1))
    M = np.abs(T).max(); worst = max(worst, np.abs(dT).max() / M); worst2 = max(worst2, ((dT / M)**2 + n * n * (T / M)**2).max() / n**2)
print(round(worst, 3), "<=", n, "  Bernstein-Szego ratio", round(worst2, 4), "<= 1")

**Problem 10.20 (absolute convergence).** The Rudin--Shapiro function: Hölder ratio $|f(x+h)-f(x)|/|h|^{1/2}$ stays bounded, while $\sum|c_k|$ grows
like the number of blocks.

In [ ]:
def rudin_shapiro(J):
    P, Q = [np.array([1.0])], [np.array([1.0])]
    for k in range(J):
        P.append(np.concatenate([P[-1], Q[-1]])); Q.append(np.concatenate([P[-2], -Q[-1]]))
    return P
J = 12; Ps = rudin_shapiro(J)
coef = np.zeros(2**(J + 1), complex)
for j in range(J + 1): coef[2**j:2**(j + 1)] = Ps[j] / 2**j
f = lambda x: np.sum(coef * np.exp(1j * np.arange(len(coef)) * x))
xs = rng.random(20) * 2 * pi
print(max(abs(f(x + h) - f(x)) / sqrt(h) for x in xs for h in (1e-1, 1e-2, 1e-3)), np.abs(coef).sum(), J + 1)
print("sup norms vs sqrt(2^(j+1)):", [round(np.abs(np.polyval(Ps[j][::-1], np.exp(1j * np.linspace(0, 2 * pi, 4096)))).max() / sqrt(2**(j + 1)), 3) for j in range(1, 9)])

**Problem 10.21 (polygons).** The Fan--Taussky--Todd inequality and the area inequality for random polygons; equality for the regular one.

In [ ]:
n = 7; w = np.exp(2j * pi / n)
def lhs(z): return np.sum(np.abs(np.roll(z, -1) - z)**2)
def area(z): return 0.5 * np.imag(np.sum(np.conj(z) * np.roll(z, -1)))
ratios = []
for _ in range(2000):
    z = rng.normal(size=n) + 1j * rng.normal(size=n); x = z - z.mean()
    ratios.append((lhs(x) / np.sum(np.abs(x)**2), lhs(z) - 4 * np.tan(pi / n) * area(z)))
print(round(min(r[0] for r in ratios), 5), round(4 * sin(pi / n)**2, 5), min(r[1] for r in ratios) >= 0)
reg = w**np.arange(n); print(round(lhs(reg) - 4 * np.tan(pi / n) * area(reg), 12))

**Problem 10.22 (Siegel).** For a lattice-free ellipse in the plane, the defect $v(1-v)$ against a finite part of the Fourier sum.

In [ ]:
# K = open disc of radius 1: no non-zero integer point; v = vol(K/2) = pi/4
v = pi / 4
def coeff(k1, k2):                      # integral over the disc of radius 1/2 of e(-k.x), via Bessel J1
    r = math.hypot(k1, k2)
    return v if r == 0 else float(0.5 * mp.besselj(1, pi * r) / r)
s = sum(coeff(k1, k2)**2 for k1 in range(-40, 41) for k2 in range(-40, 41) if (k1, k2) != (0, 0))
print(round(v * (1 - v), 6), round(s, 6))

The partial Fourier sum creeps up to $v(1-v)$, as Siegel's identity (Parseval instead of Bessel) predicts: the defect $1-\frac\pi4$ of the disc is
accounted for by its Fourier coefficients.

## Part B: laboratory

**L10.1 (Sato--Tate for Kloosterman sums).** For $p=2003$, write $K(a)=2\sqrt p\cos\theta_a$ for $a\ne0$. Plot the histogram of the $\theta_a$ against the
density $\frac2\pi\sin^2\theta$ on $[0,\pi]$. Compare with the angles of $\cos(2\pi a/p)$, which are uniform.

**L10.2 (Guilbaud by simulation).** Simulate elections with $n=1001$ voters ranking three candidates at random and estimate the probability of a
Condorcet cycle. Then repeat with voters who are not uniform: say half the voters choose $a>b>c$ with probability $\frac12$ and
uniform otherwise. Does Kalai's formula still apply?

**L10.3 (the constant 9).** For Hermite polynomials $H_d$ of a standard Gaussian, compute $r_d=\mathbb E[H_d^4]/\mathbb E[H_d^2]^2$ and plot $r_d^{1/d}$ for
$d\le40$. Then check on the cube, for $n=12$, that the polynomial $\prod$ of normalised block sums with $2$ blocks is beaten by a suitable
discrete Hermite-like polynomial of degree $2$ in one block sum.

**L10.4 (Erdős's Bernoulli convolution).** For $\lambda=\frac1\phi$ with $\phi$ the golden ratio, compute $|\prod_{k\ge1}\cos(t\lambda^k)|$ along $t=2\pi\phi^m$ and along
$t=2\pi\cdot1.5^m$. Explain the difference with Problem 10.9.

## Solutions

### L10.1

In [ ]:
p = 2003; K = kloosterman(p)[1:]
theta = np.arccos(np.clip(K / (2 * sqrt(p)), -1, 1))
grid = np.linspace(0, pi, 200)
plt.figure(figsize=(6, 3)); plt.hist(theta, bins=40, density=True, alpha=0.6, label="Kloosterman angles")
plt.plot(grid, 2 / pi * np.sin(grid)**2, "r", label="Sato-Tate"); plt.legend(); plt.show()
print("max |K|/(2 sqrt p) =", round(np.abs(K).max() / (2 * sqrt(p)), 4), " mean of sin^2 theta:", round(np.mean(np.sin(theta)**2), 3), "(Sato-Tate: 3/4)")

The angles pile up in the middle, like $\sin^2\theta$, not uniformly: the Kloosterman sums of a prime behave like traces of random matrices
in $SU(2)$, Katz's theorem. All of them obey Weil's bound.

### L10.2

In [ ]:
def simulate(n, trials, bias=0.0):
    cycles = 0
    for _ in range(trials):
        r = rng.integers(0, 6, n)
        if bias: r = np.where(rng.random(n) < bias, 0, r)
        xyz = np.array([(1, 1, -1), (1, -1, -1), (-1, 1, -1), (-1, 1, 1), (1, -1, 1), (-1, -1, 1)])[r]
        s = np.sign(xyz.sum(0)); cycles += abs(s.sum()) == 3
    return cycles / trials
print(simulate(1001, 20000), 1 - (0.75 + 3 / (2 * pi) * asin(1 / 3)))
print(simulate(1001, 4000, bias=0.25))

About $8.8\%$ of elections with uniform voters have a cycle. With biased voters the cycles disappear almost completely: ranking $0$ is
$a>b>c$, and a systematic preference gives each duel a clear majority. Kalai's formula used the uniform distribution of each
voter's triple in (a); for other distributions the pairs are no longer $(-\frac13)$-correlated with uniform marginals, and the formula
changes.

### L10.3

In [ ]:
from numpy.polynomial import hermite_e as He
xg, wg = He.hermegauss(250); wg = wg / wg.sum()
ds = range(1, 41); r = []
for d in ds:
    v = He.hermeval(xg, [0] * d + [1]); r.append((wg * v**4).sum() / (wg * v**2).sum()**2)
plt.figure(figsize=(6, 3)); plt.plot(list(ds), [ri**(1 / d) for d, ri in zip(ds, r)], "o-"); plt.axhline(9, color="r"); plt.show()
n = 12; X = np.array(list(itertools.product([-1, 1], repeat=n))); L = X.sum(1) / sqrt(n)
h2 = L**2 - 1                                   # = (2/n) sum_{i<j} x_i x_j, multilinear of degree 2
blocks = (X[:, :6].sum(1) / sqrt(6)) * (X[:, 6:].sum(1) / sqrt(6))
ratio = lambda f: np.mean(f**4) / np.mean(f**2)**2
print(round(ratio(h2), 3), round(ratio(blocks), 3), "Gaussian He_2:", r[1])

$r_d^{1/d}$ climbs slowly towards $9$: for large $d$, $r_d$ is $9^d$ up to a factor that is small on the exponential scale, so the base $9$ in the
Bonami lemma is optimal. On the cube, $L^2-1=\frac2n\sum_{i<j}x_ix_j$ (a multilinear polynomial of degree $2$) has ratio $11.2$ for $n=12$, tending to the
Gaussian value $15$ of $He_2$ as $n$ grows: already more than the $(3-\frac13)^2=7.1$ of the block product.

### L10.4

In [ ]:
phi = (1 + mp.sqrt(5)) / 2; lam = 1 / phi
def cf(t, K=200): return abs(mp.fprod(mp.cos(t * lam**k) for k in range(1, K)))
print([mp.nstr(cf(2 * mp.pi * phi**m), 6) for m in (5, 10, 15, 20, 25)])
print([mp.nstr(cf(2 * mp.pi * mp.mpf(1.5)**m), 6) for m in (5, 10, 15, 20, 25)])
print([mp.nstr(abs(phi**m - mp.nint(phi**m)), 4) for m in (5, 10, 15, 20)])

Along $t=2\pi\phi^m$ the characteristic function stays near a positive constant, because $\phi^m$ is within $\phi^{-m}$ of an integer (the Lucas
numbers), so the factors $\cos(2\pi\phi^{m-k})$ with $k<m$ are all close to $1$. This is the Cantor mechanism of Problem 10.9 with $3^m$ replaced
by the almost-integers $\phi^m$: the law of $\sum\pm\phi^{-k}$ is singular (Erdős, 1939). Along $1.5^m$ the values decay.